# SA3 Playground (local GPU)

Same modes as the Colab notebook — adjust prompts and parameters freely.

| Model | Steps / cfg defaults |
|-------|----------------------|
| `medium` | steps **8**, cfg **1** (cfg has little effect) |
| `medium-base` | steps **~50**, cfg **~7** for prompt control |

**Setup:** clone [stable-audio-3](https://github.com/Stability-AI/stable-audio-3) alongside this repo, `uv sync` + Flash Attention (see README). Accept HF licenses for `medium`, `medium-base`, `SAME-L`.

In [ ]:
import os
import sys
from pathlib import Path

PLAYGROUND = Path.cwd().resolve()
SA3_DIR = Path(os.environ.get("SA3_DIR", PLAYGROUND.parent / "stable-audio-3")).resolve()

if not (SA3_DIR / "stable_audio_3").exists():
    raise RuntimeError(f"stable-audio-3 not found at {SA3_DIR}. Set SA3_DIR or clone alongside playground.")

for p in (str(SA3_DIR), str(PLAYGROUND)):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch

print("python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
try:
    import flash_attn
    print("flash_attn:", flash_attn.__version__)
except ImportError:
    print("flash_attn: not installed (required for medium / medium-base)")

## HuggingFace login

Set `HF_TOKEN` in the environment, or paste a token below. Accept licenses first.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

from hf_check import login_if_ready

hf_token = widgets.Password(
    description="HF token:",
    value=os.environ.get("HF_TOKEN", ""),
    layout=widgets.Layout(width="500px"),
)
hf_btn = widgets.Button(description="Login", button_style="primary")
hf_out = widgets.Output()


def _hf_login(_):
    with hf_out:
        hf_out.clear_output()
        try:
            whoami, checks = login_if_ready(hf_token.value)
        except ValueError as exc:
            print(exc)
            return
        except Exception as exc:
            print(f"Failed: {exc}")
            return
        for c in checks:
            mark = "OK" if c.ok else "FAIL"
            print(f"[{mark}] {c.label}")
            if not c.ok:
                print(f"       {c.message}")
        if all(c.ok for c in checks):
            print(f"\nLogged in as {whoami.get('name', '?')}.")
        else:
            print("\nNot logged in.")


hf_btn.on_click(_hf_login)
display(widgets.VBox([hf_token, hf_btn, hf_out]))

## Input audio

Set a WAV path or upload. Resampled to **44.1 kHz** stereo.

In [ ]:
from IPython.display import Audio, display as ipy_display

from audio_util import TARGET_SR, load_path, load_uploaded_wav, parse_file_upload, tensor_for_audio_widget

input_path = widgets.Text(value="input.wav", description="WAV path", layout=widgets.Layout(width="70%"))
load_path_btn = widgets.Button(description="Load path")
upload = widgets.FileUpload(accept=".wav", multiple=False, description="or upload")
upload_out = widgets.Output()

STATE = {"wav": None, "sr": TARGET_SR}


def _preview(wav, sr, label=""):
    with upload_out:
        upload_out.clear_output()
        print(f"Loaded {label}: {wav.shape[-1]/sr:.2f}s")
        mono, _ = tensor_for_audio_widget(wav, sr)
        ipy_display(Audio(mono.numpy(), rate=sr))


def _on_path(_):
    p = Path(input_path.value.strip())
    if not p.is_file():
        print(f"Not found: {p}")
        return
    wav, sr = load_path(str(p))
    STATE["wav"] = wav
    STATE["sr"] = sr
    _preview(wav, sr, p.name)


def _on_upload(change):
    name, data = parse_file_upload(upload.value)
    if data is None:
        return
    wav, sr = load_uploaded_wav(data)
    STATE["wav"] = wav
    STATE["sr"] = sr
    _preview(wav, sr, name)


load_path_btn.on_click(_on_path)
upload.observe(_on_upload, names="value")
display(widgets.VBox([input_path, load_path_btn, upload, upload_out]))

## Audio-to-Audio (`medium-base`)

| Control | API | Notes |
|---------|-----|-------|
| **prompt** | `prompt` | Edit style / genre / mood |
| **sigma** | `init_noise_level` | 0.0–1.0; lower = closer to input |
| **steps** | `steps` | ~50 for base |
| **cfg** | `cfg_scale` | ~7 for base; 1 = more freedom |

In [ ]:
from playground_core import load_model, run_ia

ia_base_prompt = widgets.Textarea(value="upbeat drum loop with clear kick", description="prompt", layout=widgets.Layout(width="90%", height="60px"))
ia_base_sigma = widgets.FloatSlider(value=0.3, min=0.01, max=0.95, step=0.01, description="sigma")
ia_base_steps = widgets.IntSlider(value=50, min=4, max=100, description="steps")
ia_base_cfg = widgets.FloatSlider(value=7.0, min=1.0, max=15.0, step=0.5, description="cfg")
ia_base_btn = widgets.Button(description="Run (medium-base)", button_style="success")
ia_base_out = widgets.Output()


def _run_ia_base(_):
    if STATE["wav"] is None:
        print("Load WAV first.")
        return
    with ia_base_out:
        ia_base_out.clear_output()
        print("Loading medium-base...")
        model = load_model("medium-base")
        print("Generating...")
        out = run_ia(model, STATE["wav"], sigma=ia_base_sigma.value, prompt=ia_base_prompt.value,
                     steps=ia_base_steps.value, cfg=ia_base_cfg.value)
        mono, sr = tensor_for_audio_widget(out, model.model.sample_rate)
        ipy_display(Audio(mono.numpy(), rate=sr))


ia_base_btn.on_click(_run_ia_base)
display(widgets.VBox([ia_base_prompt, ia_base_sigma, ia_base_steps, ia_base_cfg, ia_base_btn, ia_base_out]))

## Audio-to-Audio (`medium`)

Post-trained checkpoint: **steps=8**, **cfg=1** (fixed).

In [ ]:
ia_med_prompt = widgets.Textarea(value="upbeat drum loop with clear kick", description="prompt", layout=widgets.Layout(width="90%", height="60px"))
ia_med_sigma = widgets.FloatSlider(value=0.3, min=0.01, max=0.95, step=0.01, description="sigma")
ia_med_steps = widgets.IntSlider(value=8, min=4, max=32, description="steps")
ia_med_btn = widgets.Button(description="Run (medium)", button_style="success")
ia_med_out = widgets.Output()


def _run_ia_med(_):
    if STATE["wav"] is None:
        print("Load WAV first.")
        return
    with ia_med_out:
        ia_med_out.clear_output()
        print("Loading medium...")
        model = load_model("medium")
        out = run_ia(model, STATE["wav"], sigma=ia_med_sigma.value, prompt=ia_med_prompt.value,
                     steps=ia_med_steps.value, cfg=1.0)
        mono, sr = tensor_for_audio_widget(out, model.model.sample_rate)
        ipy_display(Audio(mono.numpy(), rate=sr))


ia_med_btn.on_click(_run_ia_med)
display(widgets.VBox([ia_med_prompt, ia_med_sigma, ia_med_steps, ia_med_btn, ia_med_out]))

## Inpainting (`medium`)

Regenerate `[mask_start, mask_end)` only; rest preserved.

In [ ]:
from playground_core import run_inpaint

inp_prompt = widgets.Textarea(value="clear percussion and kick pulse", description="prompt", layout=widgets.Layout(width="90%", height="60px"))
inp_duration = widgets.FloatSlider(value=30.0, min=5.0, max=120.0, step=1.0, description="duration")
inp_m0 = widgets.FloatSlider(value=10.0, min=0.0, max=60.0, step=0.5, description="mask_start")
inp_m1 = widgets.FloatSlider(value=20.0, min=0.5, max=120.0, step=0.5, description="mask_end")
inp_steps = widgets.IntSlider(value=8, min=4, max=32, description="steps")
inp_btn = widgets.Button(description="Run Inpaint", button_style="success")
inp_out = widgets.Output()


def _run_inp(_):
    if STATE["wav"] is None:
        print("Load WAV first.")
        return
    with inp_out:
        inp_out.clear_output()
        model = load_model("medium")
        out = run_inpaint(model, STATE["wav"], mask_start=inp_m0.value, mask_end=inp_m1.value,
                          prompt=inp_prompt.value, duration=inp_duration.value,
                          steps=inp_steps.value, cfg=1.0)
        mono, sr = tensor_for_audio_widget(out, model.model.sample_rate)
        ipy_display(Audio(mono.numpy(), rate=sr))


inp_btn.on_click(_run_inp)
display(widgets.VBox([inp_prompt, inp_duration, inp_m0, inp_m1, inp_steps, inp_btn, inp_out]))

## Continuation (`medium`)

Extend clip: `mask_start` ≈ input length, `mask_end` = `duration`.

In [ ]:
cont_prompt = widgets.Textarea(value="dreamy synth pad, gradual build", description="prompt", layout=widgets.Layout(width="90%", height="60px"))
cont_duration = widgets.FloatSlider(value=30.0, min=5.0, max=120.0, step=1.0, description="duration")
cont_m0 = widgets.FloatSlider(value=15.0, min=0.0, max=60.0, step=0.5, description="mask_start")
cont_m1 = widgets.FloatSlider(value=30.0, min=0.5, max=120.0, step=0.5, description="mask_end")
cont_steps = widgets.IntSlider(value=8, min=4, max=32, description="steps")
cont_btn = widgets.Button(description="Run Continuation", button_style="success")
cont_out = widgets.Output()


def _run_cont(_):
    if STATE["wav"] is None:
        print("Load WAV first.")
        return
    with cont_out:
        cont_out.clear_output()
        model = load_model("medium")
        out = run_inpaint(model, STATE["wav"], mask_start=cont_m0.value, mask_end=cont_m1.value,
                          prompt=cont_prompt.value, duration=cont_duration.value,
                          steps=cont_steps.value, cfg=1.0)
        mono, sr = tensor_for_audio_widget(out, model.model.sample_rate)
        ipy_display(Audio(mono.numpy(), rate=sr))


cont_btn.on_click(_run_cont)
display(widgets.VBox([cont_prompt, cont_duration, cont_m0, cont_m1, cont_steps, cont_btn, cont_out]))

## Inversion — Sanity Check (`medium-base`)

Round-trip: input ≈ reconstructed (prompt="", gamma=0, eta=0).

In [ ]:
from playground_core import run_inversion_sanity

inv_sanity_btn = widgets.Button(description="Run Sanity Check", button_style="warning")
inv_sanity_out = widgets.Output()


def _run_inv_sanity(_):
    if STATE["wav"] is None:
        print("Load WAV first.")
        return
    with inv_sanity_out:
        inv_sanity_out.clear_output()
        print("Loading medium-base...")
        model = load_model("medium-base")
        print("Inverting + reconstructing...")
        out = run_inversion_sanity(model, STATE["wav"])
        mono_in, sr = tensor_for_audio_widget(STATE["wav"], STATE["sr"])
        mono_out, _ = tensor_for_audio_widget(out, model.model.sample_rate)
        print("Input:")
        ipy_display(Audio(mono_in.numpy(), rate=sr))
        print("Reconstructed:")
        ipy_display(Audio(mono_out.numpy(), rate=sr))


inv_sanity_btn.on_click(_run_inv_sanity)
display(widgets.VBox([inv_sanity_btn, inv_sanity_out]))

## Inversion — Edit (`medium-base`)

Two-phase RF-Inversion: invert → denoise toward **prompt** with gamma / eta / s / tau.

In [ ]:
from playground_core import run_inversion_edit

inv_prompt = widgets.Textarea(value="upbeat drum loop with clear kick", description="prompt", layout=widgets.Layout(width="90%", height="60px"))
inv_gamma = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description="gamma")
inv_eta = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description="eta")
inv_s = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description="s")
inv_tau = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description="tau")
inv_cfg = widgets.FloatSlider(value=7.0, min=1.0, max=15.0, step=0.5, description="cfg")
inv_steps_a = widgets.IntSlider(value=50, min=4, max=100, description="inv_steps")
inv_steps_b = widgets.IntSlider(value=50, min=4, max=100, description="samp_steps")
inv_edit_btn = widgets.Button(description="Run Edit Inversion", button_style="success")
inv_edit_out = widgets.Output()


def _run_inv_edit(_):
    if STATE["wav"] is None:
        print("Load WAV first.")
        return
    with inv_edit_out:
        inv_edit_out.clear_output()
        model = load_model("medium-base")
        print("Running invert → edit...")
        out = run_inversion_edit(
            model, STATE["wav"], prompt=inv_prompt.value,
            gamma=inv_gamma.value, eta=inv_eta.value,
            s=inv_s.value, tau=inv_tau.value, cfg=inv_cfg.value,
            inv_steps=inv_steps_a.value, samp_steps=inv_steps_b.value,
        )
        mono, sr = tensor_for_audio_widget(out, model.model.sample_rate)
        print("Edited:")
        ipy_display(Audio(mono.numpy(), rate=sr))


inv_edit_btn.on_click(_run_inv_edit)
display(widgets.VBox([
    inv_prompt, inv_gamma, inv_eta, inv_s, inv_tau, inv_cfg,
    inv_steps_a, inv_steps_b, inv_edit_btn, inv_edit_out,
]))